# Text Representation

## Basic Vectorization
The Twitter dataset (`tweets.csv`) was collected in February of 2015. Contributors were asked to first classify positive, negative, and neutral tweets, followed by categorizing negative reasons (such as "late flight" or "rude service"). The dataset can be found [here.](https://www.kaggle.com/crowdflower/twitter-airline-sentiment)

You should build an NLP pipeline to find all tweets that are related to `bad catering service`. In particular, you should do the following:
- Load the `tweets` dataset using [Pandas](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html). You can find this dataset in the datasets folder.
- Train a text representation model, such as the [bag of n-gram vectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) or [TF-IDF vectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html), on the content of the tweets.
- Apply the trained text representation model to vectorize the query (i.e., `bad catering service`) and all documents (i.e., tweets).
- Calculate the similarity of each vectorized tweet to the vectorized query using a similarity measure, such as [cosine similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html).
- Rank the tweets based on the similarity of their vectors to the query vector.
- Check the documentation to identify the most important hyperparameters, attributes, and methods of the model. Use them in practice.

## Advanced Vectorization
You should build an NLP pipeline using a pretrained word embedding model. In particular, you should do the following:
- Load a pretrained word embedding model, such as word2vec or glove, using [gensim](https://radimrehurek.com/gensim/downloader.html).
- Examine the word vectors. For example, what is the most similar word to `Berlin`?
- Visualize a sample of word vectors using necessary libraries, such as [Sklearn's t-SNE](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html) and [Plotly](https://plotly.com/python/line-and-scatter/).

In [39]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer #to convert text to vector
from sklearn.metrics.pairwise import cosine_similarity #for find similarity of 2 vectors

In [52]:
url = "https://raw.githubusercontent.com/m-mahdavi/teaching/refs/heads/main/datasets/tweets.csv"
df = pd.read_csv(url)
df.head(5)

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [41]:
#convert Nan to "" , we need list in Tf-IDf
tweets = df['text'].fillna('').tolist()


In [63]:
#Train TF-IDF
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2) # unigram + bigram
    ,stop_words='english', #remove the,is,a
    lowercase=True) # all words in lowercase

tweet_vectors = vectorizer.fit_transform(tweets)# train and fit vocabs from tweeter


In [54]:
#Vectorize query
query = "bad catering service"
query_vector = vectorizer.transform([query])

In [55]:
#Cosine similarity
similarities = cosine_similarity(query_vector, tweet_vectors).flatten()

In [56]:
#Rank tweets
df['similarity'] = similarities #add similarities to table
top_tweets = df.sort_values('similarity', ascending=False).head(10) #ranking
print(top_tweets[['text', 'similarity']])

                                                    text  similarity
14570  @AmericanAir I really hope it departs. They sa...    0.307858
14486  @AmericanAir first the pilot, then the caterin...    0.243568
14156  @AmericanAir how hard is it to have catering r...    0.191104
11635  @USAirways bad weather shouldn't mean bad service    0.187078
11123  @USAirways your app is bad, and you should fee...    0.173993
10923  @USAirways has completely wasted my work day! ...    0.150807
1720   @united and I was denied an upgrade because of...    0.150223
12737    @AmericanAir SO BAD service in Miami, AirPort..    0.148351
13353       @AmericanAir you're almost as bad as @united    0.142554
10928  @USAirways that the flight was delayed &amp; c...    0.138942


In [12]:
! pip install gensim
! pip install plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 51.9 MB/s eta 0:00:00


In [26]:
import gensim.downloader as api
import numpy as np
from sklearn.manifold import TSNE
import plotly.express as px


In [59]:
#Load pretrained Word2Vec
model = api.load('glove-wiki-gigaword-100')

[==================================================] 100.0% 128.1/128.1MB downloaded


In [80]:
#Most similar to Berlin

print(model.most_similar('berlin'))

[('munich', 0.8067100048065186), ('vienna', 0.7942332029342651), ('hamburg', 0.7488428354263306), ('warsaw', 0.7330315113067627), ('bonn', 0.731087863445282), ('germany', 0.7289800047874451), ('prague', 0.7246278524398804), ('dresden', 0.722325325012207), ('frankfurt', 0.7162716388702393), ('cologne', 0.712157666683197)]


In [79]:
#t-SNE visualization
words = ['Berlin', 'Paris', 'London', 'Tokyo',
         'dog', 'cat', 'horse',
         'king', 'queen', 'man', 'woman']

In [82]:
#lowercas
words_lower = [w.lower() for w in words]
vectors =np.array([ model[w] for w in words_lower])
tsne =TSNE(n_components=2, random_state=42,perplexity=8)
vectors_2d = tsne.fit_transform(vectors)

#Plot
fig = px.scatter(
    x=vectors_2d[:, 0],
    y=vectors_2d[:, 1],
    text=words,
    title='Word Embeddings Visualization'
)
fig.show()